# Proyek Akhir Kriptografi — AES + S-Box Hasil Algoritma Genetika

Notebook ini menjalankan pipeline proyek dari repository GitHub:

- membangun/memuat **S-box 16×16** hasil algoritma genetika,
- mengevaluasi enam kriteria S-box (NL, SAC, BIC-NL, BIC-SAC, LP, DAP),
- memvalidasi implementasi **AES-128** dengan FIPS-197,
- mengenkripsi/mendekripsi citra Lena, Mandrill, dan Landsat (ECB/CBC),
- menghitung entropy, NPCR, UACI, korelasi, MSE, dan hash,
- menampilkan hasil benchmark yang tersimpan di repository.

**Cara pakai:** `Runtime` → `Run all`. Tidak perlu GPU.  
Aplikasi Streamlit: https://applgvqz7d6ba4khl98my.streamlit.app/


In [ ]:
import os
import sys
import importlib.util

REPO_URL = "https://github.com/KiralNevan/ENKRIPSI-CITRA-DRONE-PENGINDERAAN-JAUH-UNTUK-PEMETAAN-LAHAN-PERKEBUNAN-SAWIT-DAN-MITIGASI-BENCANA-.git"
REPO_DIR = "S-BOX-Project"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())

# Colab biasanya sudah menyediakan paket-paket ini.
# Install hanya jika benar-benar belum ada.
missing = []
for module, package in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("numba", "numba"),
    ("Crypto", "pycryptodome"),
]:
    if importlib.util.find_spec(module) is None:
        missing.append(package)

if missing:
    !pip install -q {' '.join(sorted(set(missing)))}

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Working directory:", os.getcwd())
print("NumPy:", np.__version__)


## 1. S-box dan enam kriteria

S-box terbaik dari eksperimen GA dimuat dari `results/sbox/best_sbox.csv`.
Sebagai pembanding, S-box AES standar dihitung dengan fungsi metrik yang sama.


In [ ]:
from IPython.display import display, Image

from src.sbox.io import load_sbox_csv
from src.sbox.metrics import evaluate_sbox, sbox_to_grid
from src.aes.constants import AES_SBOX

ga_sbox = load_sbox_csv("results/sbox/best_sbox.csv")
aes_m = evaluate_sbox(AES_SBOX)
ga_m = evaluate_sbox(ga_sbox)

metrics = ["nl", "sac", "bic_nl", "bic_sac", "lp", "dap"]
df = pd.DataFrame(
    [
        {"S-box": "AES standar", **{k: aes_m[k] for k in metrics}},
        {"S-box": "GA terbaik", **{k: ga_m[k] for k in metrics}},
    ]
)
display(df)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sbox_to_grid(ga_sbox), cmap="viridis", interpolation="nearest")
ax.set_title("S-box 16×16 hasil GA")
ax.set_xlabel("Kolom")
ax.set_ylabel("Baris")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Nilai")
plt.show()

display(Image(filename="results/figures/ga_convergence.png", width=850))
display(Image(filename="results/figures/sbox_metrics.png", width=850))


## 2. Validasi AES-128 (FIPS-197)

Implementasi AES diuji dengan vektor uji resmi FIPS-197 Appendix C.1.
Hasil yang diharapkan: `69c4e0d86a7b0430d8cdb78070b4c55a`.


In [ ]:
from src.aes.constants import inverse_sbox
from src.aes.core import expand_key, encrypt_blocks, decrypt_blocks

key = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
plaintext = bytes.fromhex("00112233445566778899aabbccddeeff")
expected = bytes.fromhex("69c4e0d86a7b0430d8cdb78070b4c55a")

round_keys = expand_key(key, AES_SBOX)
ciphertext = encrypt_blocks(
    np.frombuffer(plaintext, dtype=np.uint8).reshape(1, 16), round_keys, AES_SBOX
).tobytes()
decrypted = decrypt_blocks(
    np.frombuffer(ciphertext, dtype=np.uint8).reshape(1, 16),
    round_keys,
    inverse_sbox(AES_SBOX),
).tobytes()

print("Ciphertext :", ciphertext.hex())
print("FIPS-197   :", ciphertext == expected)
print("Dekripsi   :", decrypted == plaintext)
assert ciphertext == expected and decrypted == plaintext


## 3. Demo enkripsi dan dekripsi citra

Demo memakai citra Lena dengan S-box GA dan mode CBC. NPCR/UACI dihitung
dengan mengubah satu piksel plaintext, lalu membandingkan dua ciphertext.


In [ ]:
from src.image.io import load_image
from src.image.metrics import image_metrics, npcr_uaci
from src.aes.modes import encrypt_image_array, decrypt_image_bytes, cipher_to_image

KEY = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
IV = bytes.fromhex("101112131415161718191a1b1c1d1e1f")


def demo(image_name="lena", algorithm="ga", mode="CBC"):
    path = "data/processed/remote_512.png" if image_name == "remote" else f"data/processed/{image_name}.png"
    image = load_image(path, None)
    sbox = ga_sbox if algorithm == "ga" else AES_SBOX

    cipher, _ = encrypt_image_array(image, KEY, mode=mode, sbox=sbox, iv=IV)
    decrypted = decrypt_image_bytes(cipher, KEY, image.shape, mode=mode, sbox=sbox, iv=IV)
    cipher_img = cipher_to_image(cipher, image.shape)
    m = image_metrics(image, cipher_img, decrypted)

    image2 = image.copy()
    image2[0, 0, 0] ^= 1
    cipher2, _ = encrypt_image_array(image2, KEY, mode=mode, sbox=sbox, iv=IV)
    npcr, uaci = npcr_uaci(cipher, cipher2)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, img, title in zip(
        axes,
        [image, cipher_img, decrypted],
        ["Plaintext", "Ciphertext", "Hasil dekripsi"],
    ):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.show()

    print(f"Entropy ciphertext : {m['entropy_cipher']:.4f}")
    print(f"NPCR (plaintext)   : {npcr:.2f}%")
    print(f"UACI (plaintext)   : {uaci:.2f}%")
    print(f"Korelasi ciphertext: H={m['corr_cipher_h']:.4f}, V={m['corr_cipher_v']:.4f}, D={m['corr_cipher_d']:.4f}")
    print(f"MSE / lossless     : {m['mse']:.4f} / {m['hash_match']}")
    return m, npcr, uaci


m, npcr, uaci = demo("lena", "ga", "CBC")


### Perbandingan cepat AES vs S-box GA pada Lena

Tabel berikut menjalankan empat konfigurasi (AES/GA × ECB/CBC) pada citra Lena.


In [ ]:
from src.image.metrics import image_metrics, npcr_uaci
from src.aes.modes import encrypt_image_array, decrypt_image_bytes, cipher_to_image

image = load_image("data/processed/lena.png", None)
rows = []

for algorithm in ["AES", "GA"]:
    sbox = AES_SBOX if algorithm == "AES" else ga_sbox
    for mode in ["ECB", "CBC"]:
        cipher, _ = encrypt_image_array(image, KEY, mode=mode, sbox=sbox, iv=IV)
        decrypted = decrypt_image_bytes(cipher, KEY, image.shape, mode=mode, sbox=sbox, iv=IV)
        cipher_img = cipher_to_image(cipher, image.shape)
        m = image_metrics(image, cipher_img, decrypted)

        image2 = image.copy()
        image2[0, 0, 0] ^= 1
        cipher2, _ = encrypt_image_array(image2, KEY, mode=mode, sbox=sbox, iv=IV)
        npcr, uaci = npcr_uaci(cipher, cipher2)

        rows.append(
            {
                "Algoritma": algorithm,
                "Mode": mode,
                "Entropy": round(m["entropy_cipher"], 4),
                "NPCR (%)": round(npcr, 2),
                "UACI (%)": round(uaci, 2),
                "MSE": m["mse"],
                "Lossless": m["hash_match"],
            }
        )

display(pd.DataFrame(rows))


## 4. Hasil benchmark yang tersimpan

CSV hasil eksperimen dan figur tersedia di folder `results/`.


In [ ]:
bench = pd.read_csv("results/metrics/benchmark_results.csv")
remote = pd.read_csv("results/metrics/remote_results_512.csv")

cols = ["image", "algorithm", "mode", "entropy", "npcr", "uaci", "time_enc_s", "mse", "hash_match"]
display(bench[cols])
display(remote[cols])

for figure in ["benchmark_entropy.png", "benchmark_npcr.png", "benchmark_uaci.png"]:
    display(Image(filename=f"results/figures/{figure}", width=850))


## 5. Demo GA cepat (opsional)

Sel pada bagian ini menjalankan GA mini (populasi 20, 20 generasi) hanya untuk
menunjukkan proses evolusi. S-box final sudah tersedia di repository.

Untuk menjalankan eksperimen GA penuh (sekitar beberapa menit):

```bash
!python -m src.experiments.run_ga --population 80 --generations 300 --stagnation 100
```


In [ ]:
from src.sbox.ga import GAConfig, genetic_algorithm

quick = genetic_algorithm(
    GAConfig(population_size=20, generations=20, seed=42),
    verbose=True,
)
print("\nMetrik S-box GA mini:")
print(quick["metrics"])


## 6. Streamlit dan repository

- **Aplikasi live:** https://applgvqz7d6ba4khl98my.streamlit.app/
- **Repository:** https://github.com/KiralNevan/ENKRIPSI-CITRA-DRONE-PENGINDERAAN-JAUH-UNTUK-PEMETAAN-LAHAN-PERKEBUNAN-SAWIT-DAN-MITIGASI-BENCANA-

Aplikasi Streamlit tidak perlu dijalankan di Colab karena sudah dideploy.
Kalau ingin menjalankan eksperimen lain:

```bash
!python -m src.experiments.run_benchmark
!python -m src.experiments.run_remote --size 512
!python -m src.experiments.make_figures
```
